# Visualização de Resultados Modal - Regressão Linear (r²)

Notebook para gerar plots dos resultados da análise de regressão linear (experimento 03), espelhando a estrutura do notebook PREDEP para comparação direta.

In [4]:
import sys
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.gridspec import GridSpec

sys.path.insert(0, str(Path.cwd().parent))

## 1. Funções de Plot

In [5]:
# Paleta de cores por trimestre climatico
SEASON_COLORS = {
    'DJF': '#4575b4',
    'MAM': '#91cf60',
    'JJA': '#d73027',
    'SON': '#fc8d59',
}
SEASON_MONTH_MAP = {
    'DJF': [12, 1, 2], 'MAM': [3, 4, 5],
    'JJA': [6, 7, 8],  'SON': [9, 10, 11],
}
REF_MONTH = {'DJF': 1, 'MAM': 4, 'JJA': 7, 'SON': 10}


def lag_to_season(lag, reference_season):
    """Retorna o trimestre ao qual um lag corresponde."""
    target = REF_MONTH[reference_season] - lag
    while target <= 0:
        target += 12
    for s, months in SEASON_MONTH_MAP.items():
        if target in months:
            return s


def make_intensity_cmap(threshold=0.1, vmax=0.6, n=256):
    """Cinza para [0, threshold]; amarelo-laranja-vermelho para [threshold, vmax]."""
    from matplotlib.colors import LinearSegmentedColormap, ListedColormap
    n_gray = max(1, round(n * threshold / vmax))
    n_hot  = n - n_gray
    yor = LinearSegmentedColormap.from_list(
        'yor', ['#ffffb2', '#fecc5c', '#fd8d3c', '#f03b20', '#bd0026']
    )
    gray = np.tile([0.75, 0.75, 0.75, 1.0], (n_gray, 1))
    hot  = yor(np.linspace(0, 1, n_hot))
    return ListedColormap(np.vstack([gray, hot]))


def plot_season_maps(data_sources, mov_index, output_dir, seasons,
                     threshold=0.1, vmax=0.6):
    """
    Mapas de intensidade e lag-por-trimestre para um ou mais datasets.

    Parameters
    ----------
    data_sources : list of (label, dict{season: array(lag, lat, lon)})
        Ex.: [('Regressao', r2_dict), ('PREDEP', alpha_dict)]
    threshold : float  -- cinza abaixo deste valor
    vmax      : float  -- limite superior da escala
    """
    from matplotlib.colors import ListedColormap, BoundaryNorm
    from matplotlib.patches import Patch

    n_src  = len(data_sources)
    n_rows = n_src * 2
    n_cols = len(seasons)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4.5 * n_cols, 4.0 * n_rows),
        squeeze=False
    )

    cmap_int     = make_intensity_cmap(threshold, vmax)
    season_order = ['DJF', 'MAM', 'JJA', 'SON']
    s_to_idx     = {s: i for i, s in enumerate(season_order)}
    cmap_lag     = ListedColormap([SEASON_COLORS[s] for s in season_order])
    norm_lag     = BoundaryNorm(np.arange(-0.5, 4), ncolors=4)
    im_int_ref   = None

    for src_i, (src_label, by_season) in enumerate(data_sources):
        row_int = src_i * 2
        row_lag = src_i * 2 + 1

        for col, season in enumerate(seasons):
            r2 = by_season.get(season)
            if r2 is None:
                axes[row_int, col].axis('off')
                axes[row_lag, col].axis('off')
                continue

            r2_max = np.nanmax(r2, axis=0)

            # linha de intensidade
            ax0 = axes[row_int, col]
            im_int = ax0.imshow(
                np.clip(r2_max, 0, vmax),
                cmap=cmap_int, vmin=0, vmax=vmax,
                aspect='auto', origin='lower', interpolation='nearest'
            )
            im_int_ref = im_int
            ax0.axis('off')
            if src_i == 0:
                ax0.set_title(season, fontsize=13, fontweight='bold', pad=6)
            if col == 0:
                ax0.text(-0.04, 0.5, src_label,
                         transform=ax0.transAxes, rotation=90,
                         va='center', ha='right', fontsize=10, fontweight='bold')

            # linha de lag por trimestre
            ax1 = axes[row_lag, col]
            best_lag = np.nanargmax(r2, axis=0).astype(float)
            best_lag[r2_max < threshold] = np.nan

            lag_grid = np.full(best_lag.shape, np.nan)
            for lv in range(r2.shape[0]):
                mask = best_lag == lv
                if mask.any():
                    lag_grid[mask] = s_to_idx[lag_to_season(lv, season)]

            ax1.imshow(lag_grid, cmap=cmap_lag, norm=norm_lag,
                       aspect='auto', origin='lower', interpolation='nearest')
            ax1.axis('off')
            if col == 0:
                ax1.text(-0.04, 0.5, f'{src_label}\nlag otimo',
                         transform=ax1.transAxes, rotation=90,
                         va='center', ha='right', fontsize=10, fontweight='bold')

    # colorbar de intensidade
    cbar = fig.colorbar(
        im_int_ref,
        ax=axes[::2, :].ravel().tolist(),
        orientation='vertical', fraction=0.015, pad=0.02,
        ticks=np.round(np.arange(0, vmax + 0.01, 0.1), 1)
    )
    cbar.set_label('Intensidade', fontsize=11)

    # legenda dos trimestres
    legend_els = [Patch(facecolor=SEASON_COLORS[s], label=s) for s in season_order]
    fig.legend(handles=legend_els, loc='lower center', ncol=4,
               fontsize=11, frameon=False, bbox_to_anchor=(0.45, -0.01))

    plt.suptitle(mov_index, fontsize=16, fontweight='bold', y=1.01)
    fig.tight_layout()

    out = output_dir / f'{mov_index}_maps.png'
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return out


print('Novas funcoes de plot carregadas.')


Novas funcoes de plot carregadas.


## 2. Processar Todos os Índices

In [6]:
from IPython.display import Image, display

SEASONS          = ['DJF', 'MAM', 'JJA', 'SON']
BASE_DIR         = Path('../results/regression')
PREDEP_DIR       = Path('../results/predep')
N_LAGS_EXPECTED  = 12


def find_season_file(mov_dir, mov_index, season):
    matches = list(mov_dir.rglob(f'{mov_index}_{season}_regression_brazil.nc'))
    return matches[0] if matches else None


def load_predep_season(mov_index, season):
    """Carrega PREDEP alpha como array (lag, lat, lon) ou None."""
    nc = PREDEP_DIR / mov_index / f'{mov_index}_{season}_xavier_brazil.nc'
    if not nc.exists():
        return None
    try:
        ds = xr.open_dataset(nc)
        lag_vars = sorted(
            [v for v in ds.data_vars if v.startswith('alpha_lag')],
            key=lambda x: int(x.split('_lag')[1])
        )
        stack = np.stack([np.abs(ds[v].values) for v in lag_vars], axis=0)
        ds.close()
        return stack
    except OSError:
        return None


mov_indices = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir()])
print(f'Indices disponiveis: {len(mov_indices)}')
for m in mov_indices:
    print(f'  {m}: {len(list((BASE_DIR / m).rglob("*.nc")))} .nc')
print()

for mov_index in mov_indices:
    results_dir = BASE_DIR / mov_index
    print(f"\n{'='*50}")
    print(f'Processando: {mov_index}')
    print('='*50)

    reg_by_season, predep_by_season = {}, {}
    lats = lons = None

    for season in SEASONS:
        nc_file = find_season_file(results_dir, mov_index, season)
        if nc_file is None:
            print(f'  {season}: nao encontrado, pulando')
            continue
        try:
            ds = xr.open_dataset(nc_file)
            lag_vars = sorted(
                [v for v in ds.data_vars if v.startswith('r2_lag')],
                key=lambda x: int(x.replace('r2_lag', ''))
            )
            if len(lag_vars) < N_LAGS_EXPECTED:
                print(f'  {season}: incompleto ({len(lag_vars)}/{N_LAGS_EXPECTED} lags), pulando')
                ds.close()
                continue
            r2_stack = np.stack([ds[v].values for v in lag_vars], axis=0)
            reg_by_season[season] = r2_stack
            if lats is None:
                lats = ds.latitude.values
                lons = ds.longitude.values
            ds.close()
            print(f'  {season}: OK -- {nc_file.relative_to(BASE_DIR)}')
        except OSError:
            print(f'  {season}: corrompido, pulando')
            continue

        predep_stack = load_predep_season(mov_index, season)
        if predep_stack is not None:
            predep_by_season[season] = predep_stack

    if not reg_by_season:
        print(f'  Nenhum dado completo para {mov_index}, pulando.')
        continue

    seasons_loaded = [s for s in SEASONS if s in reg_by_season]

    data_sources = [('Regressao (r2)', reg_by_season)]
    if predep_by_season:
        data_sources.append(('PREDEP (|alpha|)', predep_by_season))
        print(f'  PREDEP disponivel para: {list(predep_by_season.keys())}')

    output_dir = results_dir / 'plots'
    output_dir.mkdir(exist_ok=True)
    print(f'\nGerando plot -> {output_dir}/')

    out = plot_season_maps(data_sources, mov_index, output_dir, seasons_loaded)
    print(f'  {out.name}')
    display(Image(filename=str(out)))

print('\n' + '='*50)
print('Todos os indices processados.')


Indices disponiveis: 23
  AAO: 5 .nc
  AMO: 4 .nc
  AO: 4 .nc
  EPNP: 4 .nc
  IASAS: 4 .nc
  ITSMRG2: 0 .nc
  MEI: 4 .nc
  NAO: 4 .nc
  NIN03: 4 .nc
  NIN034: 4 .nc
  NIN04: 4 .nc
  NIN12: 3 .nc
  ONI: 4 .nc
  PDO: 4 .nc
  PNA: 4 .nc
  QBO: 3 .nc
  SAODI: 4 .nc
  SASDI: 4 .nc
  SIOD: 4 .nc
  SOI: 4 .nc
  TNA: 4 .nc
  TSA: 3 .nc
  WP: 4 .nc


Processando: AAO
  DJF: OK -- AAO/AAO/AAO_DJF_regression_brazil.nc
  MAM: OK -- AAO/AAO/AAO_MAM_regression_brazil.nc
  JJA: OK -- AAO/AAO/AAO_JJA_regression_brazil.nc
  SON: OK -- AAO/AAO/AAO_SON_regression_brazil.nc

Gerando plot -> ../results/regression/AAO/plots/


/tmp/ipykernel_2113783/3203768202.py:81: RuntimeWarning: All-NaN slice encountered
  r2_max = np.nanmax(r2, axis=0)


ValueError: All-NaN slice encountered